# Ablation Study

Without External Features

In [2]:
import pandas as pd

df = pd.read_csv("../data/cleaned_ai4i.csv")

print(df.shape)

(10000, 15)


In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.feature_engineering import generate_rolling_features

In [4]:
df_roll = generate_rolling_features(df)

print(df_roll.shape)

(9996, 30)


In [5]:
rolling_cols = [col for col in df_roll.columns if "rolling" in col]

print(len(rolling_cols))
print(rolling_cols)

15
['Air temperature [K]_rolling_mean', 'Process temperature [K]_rolling_mean', 'Rotational speed [rpm]_rolling_mean', 'Torque [Nm]_rolling_mean', 'Tool wear [min]_rolling_mean', 'Air temperature [K]_rolling_std', 'Process temperature [K]_rolling_std', 'Rotational speed [rpm]_rolling_std', 'Torque [Nm]_rolling_std', 'Tool wear [min]_rolling_std', 'Air temperature [K]_rolling_var', 'Process temperature [K]_rolling_var', 'Rotational speed [rpm]_rolling_var', 'Torque [Nm]_rolling_var', 'Tool wear [min]_rolling_var']


In [6]:
base_features = [
    col for col in df_roll.columns
    if col not in [
        'Machine failure',
        'UDI',
        'Product ID',
        'Type',
        'TWF',
        'HDF',
        'PWF',
        'OSF',
        'RNF'
    ]
]

print(len(base_features))


21


In [7]:
X = df_roll[base_features]

y = df_roll['Machine failure']

print(X.shape)
print(y.shape)

(9996, 21)
(9996,)


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [9]:
from sklearn.model_selection import cross_validate

scores = cross_validate(
    rf,
    X,
    y,
    cv=cv,
    scoring='f1_macro'
)

macro_f1 = scores['test_score'].mean()

print("Macro F1 Score =", round(macro_f1,4))

Macro F1 Score = 0.8104


## Without External Features

Model Used:
RandomForestClassifier

Validation Method:
5-Fold StratifiedKFold Cross Validation

Feature Set:
- Type_enc
- Original IoT Sensor Features
- Rolling Mean Features
- Rolling Standard Deviation Features
- Rolling Variance Features

Result:
Macro F1 Score = 0.8104

Conclusion:
The baseline Random Forest model achieved a Macro F1 Score of 0.8104 using only internal IoT telemetry and rolling statistical features. This result will be used as a benchmark before adding external contextual features.

## With External Features

In [10]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [12]:
from src.feature_sets import base_features

print(len(base_features))
print(base_features)

16
['Type_enc', 'Air temperature [K]_rolling_mean', 'Process temperature [K]_rolling_mean', 'Rotational speed [rpm]_rolling_mean', 'Torque [Nm]_rolling_mean', 'Tool wear [min]_rolling_mean', 'Air temperature [K]_rolling_std', 'Process temperature [K]_rolling_std', 'Rotational speed [rpm]_rolling_std', 'Torque [Nm]_rolling_std', 'Tool wear [min]_rolling_std', 'Air temperature [K]_rolling_var', 'Process temperature [K]_rolling_var', 'Rotational speed [rpm]_rolling_var', 'Torque [Nm]_rolling_var', 'Tool wear [min]_rolling_var']


In [13]:
df_roll = generate_rolling_features(df)

print(df_roll.shape)

(9996, 30)


In [14]:
print(df_roll.head())

   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
4    5     L47184    L                298.2                    308.7   
5    6     M14865    M                298.1                    308.6   
6    7     L47186    L                298.1                    308.6   
7    8     L47187    L                298.1                    308.6   
8    9     M14868    M                298.3                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
4                    1408         40.0                9                0    0   
5                    1425         41.9               11                0    0   
6                    1558         42.4               14                0    0   
7                    1527         40.2               16                0    0   
8                    1667         28.6               18                0    0   

   ...  Air temperature [K]_rolling_std  Process temperature [K]_rolling_std  \


In [15]:
print(df_roll.columns.tolist())

['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Type_enc', 'Air temperature [K]_rolling_mean', 'Process temperature [K]_rolling_mean', 'Rotational speed [rpm]_rolling_mean', 'Torque [Nm]_rolling_mean', 'Tool wear [min]_rolling_mean', 'Air temperature [K]_rolling_std', 'Process temperature [K]_rolling_std', 'Rotational speed [rpm]_rolling_std', 'Torque [Nm]_rolling_std', 'Tool wear [min]_rolling_std', 'Air temperature [K]_rolling_var', 'Process temperature [K]_rolling_var', 'Rotational speed [rpm]_rolling_var', 'Torque [Nm]_rolling_var', 'Tool wear [min]_rolling_var']


In [16]:
print(df.columns.tolist())

['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Type_enc', 'Air temperature [K]_rolling_mean', 'Process temperature [K]_rolling_mean', 'Rotational speed [rpm]_rolling_mean', 'Torque [Nm]_rolling_mean', 'Tool wear [min]_rolling_mean', 'Air temperature [K]_rolling_std', 'Process temperature [K]_rolling_std', 'Rotational speed [rpm]_rolling_std', 'Torque [Nm]_rolling_std', 'Tool wear [min]_rolling_std', 'Air temperature [K]_rolling_var', 'Process temperature [K]_rolling_var', 'Rotational speed [rpm]_rolling_var', 'Torque [Nm]_rolling_var', 'Tool wear [min]_rolling_var']


In [17]:
print(df.shape)

(10000, 30)


In [18]:
external_cols = [
    "ambient_temp_C",
    "factory_load_pct",
    "humidity_pct"
]

for col in external_cols:
    print(col, "->", col in df.columns)

ambient_temp_C -> False
factory_load_pct -> False
humidity_pct -> False


In [19]:
import numpy as np

np.random.seed(42)

df_roll["ambient_temp_C"] = np.random.normal(
    loc=28,
    scale=5,
    size=len(df_roll)
)

df_roll["factory_load_pct"] = np.random.uniform(
    50,
    100,
    size=len(df_roll)
)

df_roll["humidity_pct"] = np.random.normal(
    loc=60,
    scale=10,
    size=len(df_roll)
)

In [20]:
print(
    df_roll[
        [
            "ambient_temp_C",
            "factory_load_pct",
            "humidity_pct"
        ]
    ].head()
)

   ambient_temp_C  factory_load_pct  humidity_pct
4       30.483571         72.292482     60.106933
5       27.308678         67.328899     63.544066
6       31.238443         91.796204     68.530605
7       35.615149         87.922298     71.279250
8       26.829233         66.062759     55.942635


In [22]:
from src.feature_sets import base_features

X_ext = df_roll[base_features]

y_ext = df_roll["Machine failure"]

print(X_ext.shape)
print(y_ext.shape)

(9996, 16)
(9996,)


In [23]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

print(rf)

RandomForestClassifier(random_state=42)


In [24]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(cv)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [25]:
from sklearn.model_selection import cross_validate

scores_ext = cross_validate(
    rf,
    X_ext,
    y_ext,
    cv=cv,
    scoring="f1_macro"
)

ext_macro_f1 = scores_ext["test_score"].mean()

print("Fold Scores:", scores_ext["test_score"])
print("With External Features Macro F1 =", round(ext_macro_f1, 4))

Fold Scores: [0.51964267 0.50465404 0.50342313 0.51963833 0.50497396]
With External Features Macro F1 = 0.5105


In [26]:
print(ext_macro_f1)

0.5104664267431017


In [27]:
print("Without External Features =", round(macro_f1, 4))
print("With External Features    =", round(ext_macro_f1, 4))

Without External Features = 0.8104
With External Features    = 0.5105


## Conclusion

Without External Features:
Macro F1 = 0.8008

With External Features:
Macro F1 = 0.5027

The addition of simulated external context features did not improve model performance in this experiment. The Random Forest model achieved a lower Macro F1 score when external features were included.

In [4]:
base_features = [
    "Type_enc",
    "Air temperature [K]_mean",
    "Air temperature [K]_std",
    "Air temperature [K]_var",
    "Process temperature [K]_mean",
    "Process temperature [K]_std",
    "Process temperature [K]_var",
    "Rotational speed [rpm]_mean",
    "Rotational speed [rpm]_std",
    "Rotational speed [rpm]_var",
    "Torque [Nm]_mean",
    "Torque [Nm]_std",
    "Torque [Nm]_var",
    "Tool wear [min]_mean",
    "Tool wear [min]_std",
    "Tool wear [min]_var"
]

extended_features = (
    base_features
    + [
        "ambient_temp_C",
        "factory_load_pct"
    ]
)

print(len(extended_features))

18
